# Library import

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image
from torchsummary import summary

import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
# ---------- CONFIG ----------
TRAIN_CSV = "/kaggle/input/digit-recognizer/train.csv"      # your training CSV (label + 784 pixels)
TEST_CSV = "/kaggle/input/digit-recognizer/test.csv"        # optional: kaggle test set (no label)


In [ ]:
BATCH_SIZE = 128
EPOCHS = 40
LR = 1e-3
WEIGHT_DECAY = 1e-4
NUM_CLASSES = 10
IMG_SIZE = 28
MODEL_OUT = "best_cnn_aug.pth"

# Dataset

In [ ]:
# ------- Dataset -------
class MNISTCSV(Dataset):
    def __init__(self, df_or_path, transform=None, is_test=False):
        if isinstance(df_or_path, str):
            df = pd.read_csv(df_or_path)
        else:
            df = df_or_path.copy()
        self.is_test = is_test
        if not is_test:
            assert 'label' in df.columns, "train csv must contain 'label' column"
            self.labels = df['label'].astype(np.int64).values
            pixels = df.drop('label', axis=1).values.astype(np.uint8)
        else:
            self.labels = None
            pixels = df.values.astype(np.uint8)
        # reshape to (N, H, W)
        self.images = pixels.reshape(-1, IMG_SIZE, IMG_SIZE)
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.fromarray(self.images[idx])  # PIL Image (mode='L')
        if self.transform:
            img = self.transform(img)
        if self.is_test:
            return img
        label = int(self.labels[idx])
        return img, label


# Transforms

In [ ]:
# Training augmentations (careful: no horizontal flip for digits)

train_transform = T.Compose([
    T.RandomRotation(15, fill=(0,)),  # rotate +/-15 degrees
    T.RandomAffine(degrees=0, translate=(0.08,0.08), scale=(0.9,1.1), shear=10, fill=0),
    T.RandomPerspective(distortion_scale=0.2, p=0.2, fill=0),
    T.ToTensor(),                      # converts to [0,1]
    T.Normalize((0.1307,), (0.3081,)), # MNIST mean/std (on 0-1 scale)
    T.RandomErasing(p=0.2, scale=(0.02,0.2))
])

val_transform = T.Compose([
    T.ToTensor(),
    T.Normalize((0.1307,), (0.3081,))
])

# Model (small but effective CNN)

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.08)
        )
    def forward(self, x):
        return self.block(x)

class CNNAugNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(1, 32),
            ConvBlock(32, 64),
            ConvBlock(64, 128)
        )
        self.pool = nn.AdaptiveAvgPool2d(1)  # global avg pool -> (B,128,1,1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = self.classifier(x)
        return x

# training and validate fn

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for imgs, labels in loader:
        imgs = imgs.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, 100.0 * correct / total



def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            
    return running_loss / total, 100.0 * correct / total


# Main training flow

In [ ]:
def main_train():
    # load CSV into dataframe
    df = pd.read_csv(TRAIN_CSV)
    train_df, val_df = train_test_split(df, test_size=0.15, stratify=df['label'], random_state=42)

    train_ds = MNISTCSV(train_df, transform=train_transform, is_test=False)
    
    val_ds = MNISTCSV(val_df, transform=val_transform, is_test=False)

    train_loader = DataLoader(train_ds, 
                              batch_size=BATCH_SIZE, 
                              shuffle=True, num_workers=2, 
                              pin_memory=True
                            )
    val_loader = DataLoader(val_ds, 
                            batch_size=BATCH_SIZE, 
                            shuffle=False, num_workers=2,
                            pin_memory=True
                        )

    # model define
    model = CNNAugNet(num_classes=NUM_CLASSES).to(device)
    # use label smoothing via CrossEntropyLoss if PyTorch version supports; else simple CE
    
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05) if hasattr(nn.CrossEntropyLoss, "__call__") else nn.CrossEntropyLoss()
    
    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    # scheduler: cosine annealing (works well)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_val_acc = 0.0
    for epoch in range(1, EPOCHS+1):
        train_loss, train_acc = train_one_epoch(
            model, 
            train_loader, 
            criterion, 
            optimizer, 
            device
        )
        val_loss, val_acc = validate(
            model, 
            val_loader, 
            criterion, 
            device
        )
        scheduler.step()

        print(f"Epoch {epoch:02d} | Train Loss {train_loss:.4f} Acc {train_acc:.2f}% | Val Loss {val_loss:.4f} Acc {val_acc:.2f}%")

        # save best
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), MODEL_OUT)
            print(f"Saved best model (val acc {best_val_acc:.2f}%) to {MODEL_OUT}")

    print("Training finished. Best val acc:", best_val_acc)

# Inference + TTA for single image (and test csv)

In [ ]:
def tta_predict_image(model, pil_img, tta_transforms, device):
    # pil_img: PIL Image (grayscale)
    model.eval()
    probs_sum = None
    with torch.no_grad():
        for tr in tta_transforms:
            img_t = tr(pil_img).unsqueeze(0).to(device)  # (1,1,28,28)
            logits = model(img_t)
            probs = torch.softmax(logits, dim=1)
            probs_sum = probs if probs_sum is None else probs_sum + probs
            
    probs_avg = probs_sum / len(tta_transforms)
    
    pred = probs_avg.argmax(dim=1).cpu().item()
    
    return pred, probs_avg.cpu().numpy()

In [ ]:
def run_tta_demo(image_path):
    # build TTA transforms (small rotations/affines)
    base = T.Compose([T.ToTensor(), T.Normalize((0.1307,), (0.3081,))])
    tta_transforms = [
        T.Compose([T.RandomRotation(0), T.ToTensor(), T.Normalize((0.1307,), (0.3081,))]),  # identity
    ]
    # add small rotations
    for ang in [-12, -8, -4, 0, 4, 8, 12]:
        tta_transforms.append(T.Compose([T.RandomRotation((ang, ang)), T.ToTensor(), T.Normalize((0.1307,), (0.3081,))]))

    # load model
    model = CNNAugNet(num_classes=NUM_CLASSES).to(device)
    model.load_state_dict(torch.load(MODEL_OUT, map_location=device))
    model.eval()

    pil = Image.open(image_path).convert("L").resize((IMG_SIZE, IMG_SIZE))
    pred, probs = tta_predict_image(model, pil, tta_transforms, device)
    print("TTA Prediction:", pred)
    print("Probs:", probs)

# Save predictions for test.csv (Kaggle submission)

In [ ]:
def predict_test_and_save(test_csv=TEST_CSV, out_csv="submission.csv"):
    if not os.path.exists(MODEL_OUT):
        raise FileNotFoundError("Model file not found. Train first or set MODEL_OUT correctly.")

    # load test
    df_test = pd.read_csv(test_csv)
    pixel_cols = [c for c in df_test.columns if c.startswith("pixel")]
    
    X = df_test[pixel_cols].values.astype(np.uint8).reshape(-1, IMG_SIZE, IMG_SIZE)
    
    test_ds = MNISTCSV(
        pd.DataFrame(X.reshape(X.shape[0], -1)), 
        transform=val_transform, 
        is_test=True
    )
    
    test_loader = DataLoader(
        test_ds, 
        batch_size=BATCH_SIZE, 
        shuffle=False
    )

    # -----------load model-----------------
    model = CNNAugNet(num_classes=NUM_CLASSES).to(device)
    model.load_state_dict(torch.load(MODEL_OUT, map_location=device))
    model.eval()

    preds = []
    with torch.no_grad():
        for imgs in tqdm(test_loader, desc="Predicting"):
            imgs = imgs.to(device)
            logits = model(imgs)
            batch_preds = logits.argmax(dim=1).cpu().numpy().tolist()
            preds.extend(batch_preds)

    # build submission
    img_ids = list(range(1, len(preds)+1))
    sub = pd.DataFrame({"ImageId": img_ids, "Label": preds})
    sub.to_csv(out_csv, index=False)
    print("Saved submission:", out_csv)


# Run training model

In [ ]:
if __name__ == "__main__":
    # 1) train
    main_train()

    # 2) TTA demo (on your drawn image)
    drawn_img_path = "/mnt/data/01.10.2025_22.29.39_REC.png"
    if os.path.exists(MODEL_OUT) and os.path.exists(drawn_img_path):
        run_tta_demo(drawn_img_path)

    # 3) If you want to create submission (uncomment)
    predict_test_and_save(TEST_CSV, out_csv="submission.csv")